# Field Shift · Punto 5: condados gemelos

**Qué ve el productor:** en la pantalla "Próxima campaña", al lado de cada opción, de 3 a 5 condados con suelo y clima parecidos al suyo que **ya hicieron** esa rotación, y cómo les fue. Por ejemplo:

> "Counties like yours that grew corn after corn: 4. Similar soil and weather doesn't guarantee the same result."

**No entrena ningún modelo.** Es una búsqueda de vecinos (kNN, *k nearest neighbors*): se mide qué tan distinto es cada condado de cada otro y se quedan los más cercanos. No es el modelo A ni el B del `CLAUDE.md`: no predice rinde.

**Se arma en dos partes:**

- **Parte 1 (pasos 0 a 10): encontrar los parecidos.** Solo con suelo y clima, que el productor no elige. La rotación y el rinde **no** entran al parecido: son justamente lo que después se mira en los gemelos.
- **Parte 2 (pasos 11 a 18): el rango de los gemelos.** De esos parecidos, quedarse con los que hicieron cada opción (`actual`, `rotado`, `continuo`, como en el punto 2), armar con sus rindes un rango p10 / mediana / p90 y compararlo con el del modelo del punto 2.

**Cómo usarla:** correr todo de arriba hacia abajo. Usa `data/ml/tabla_modelado.csv`, que deja el punto 2 y ya está en el repo.

## 0. Librerías

- **pandas / numpy:** tablas y cuentas.
- **scikit-learn:** de acá sale `nan_euclidean_distances`, que mide distancias aunque a un condado le falte algún dato.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import nan_euclidean_distances

## 1. Parámetros

Todo lo que se puede cambiar está en esta celda.

- `VARIABLES_SUELO` y `VARIABLES_CLIMA`: con qué se mide el parecido. Se eligieron para que se pueda explicar en una frase ("suelo y clima parecidos") y sin pares casi repetidos: `twi_medio` queda afuera porque se mueve igual que la pendiente (correlación 0,94), y `deficit_temporada_mm` porque se mueve casi igual que la lluvia de la temporada (0,85).
- `PESO_SUELO`: cuánto pesa el suelo frente al clima. Con 0,5 pesan lo mismo, aunque uno tenga más variables que el otro.
- `N_VECINOS`: cuántos parecidos se guardan por condado. Se guardan más de los 3 a 5 que muestra el front porque la parte 2 va a descartar los que no hicieron la opción.
- `PERCENTIL_MAXIMO`: el filtro de "parecido de verdad". Un vecino cuenta solo si está más cerca que el 10 % de los pares de condados más parecidos de toda la región. Así, un condado raro puede quedar con menos de 3 gemelos, y el front muestra confianza baja en vez de inventar parecidos.
- Solo los años 2009–2023. 2024–2025 son la prueba final del punto 2 y no se miran.

In [ ]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
SALIDA = RAIZ / "data" / "ml"
RUTA_TABLA = SALIDA / "tabla_modelado.csv"
RUTA_ENTREGA = RAIZ / "data" / "fieldshift_datos_v1.2" / "condado"   # nombres, centroides, trigo y avena

ANIOS_PRUEBA = [2024, 2025]
NOMBRES_ESTADO = {"17": "Illinois", "18": "Indiana", "19": "Iowa", "20": "Kansas"}

# Suelo: fijo, no cambia de un año a otro (SSURGO)
VARIABLES_SUELO = ["aws0_100_mm", "nccpi_maiz", "soc0_30_gm2", "pendiente_media_pct", "clase_drenaje"]
# Clima: se resume con el año típico del condado (NASA POWER)
VARIABLES_CLIMA = ["prec_temporada_mm", "gdd_temporada", "tmax_jul_c", "prec_power_mm_05"]

# Nombre en inglés para el front ("Similar in: ...") y en español para revisar
ETIQUETAS = {
    "aws0_100_mm": ("water the soil can hold", "agua que retiene el suelo"),
    "nccpi_maiz": ("soil productivity", "productividad del suelo"),
    "soc0_30_gm2": ("soil organic carbon", "carbono del suelo"),
    "pendiente_media_pct": ("slope", "pendiente"),
    "clase_drenaje": ("soil drainage", "drenaje del suelo"),
    "prec_temporada_mm": ("growing-season rain", "lluvia de la temporada"),
    "gdd_temporada": ("growing-season heat", "calor de la temporada"),
    "tmax_jul_c": ("July heat", "calor de julio"),
    "prec_power_mm_05": ("May rain", "lluvia de mayo"),
}

# El drenaje es texto: se pasa a un número ordenado (1 = drena muy mal, 7 = drena muy rápido)
ORDEN_DRENAJE = {
    "Very poorly drained": 1,
    "Poorly drained": 2,
    "Somewhat poorly drained": 3,
    "Moderately well drained": 4,
    "Well drained": 5,
    "Somewhat excessively drained": 6,
    "Excessively drained": 7,
}

PESO_SUELO = 0.5          # 0,5 = suelo y clima pesan lo mismo
N_VECINOS = 15            # parecidos que se guardan por condado
PERCENTIL_MAXIMO = 10     # un vecino cuenta si está entre el 10 % de pares más cercanos de la región
N_SIMILAR_ON = 2          # en cuántas variables se dice "Similar in: ..."
CONDADOS_EJEMPLO = ["19169", "17113"]   # Story (IA) y McLean (IL)

HAY_DATOS = RUTA_TABLA.exists()
HAY_ENTREGA = (RUTA_ENTREGA / "tabla_anual.csv").exists() and (RUTA_ENTREGA / "referencia" / "condados.csv").exists()
print("¿Está la tabla del punto 2?", HAY_DATOS, "| ¿Está la entrega v1.2?", HAY_ENTREGA)
if not HAY_DATOS:
    print("Falta data/ml/tabla_modelado.csv: correr punto2_rotaciones.ipynb. La notebook queda en espera.")
if not HAY_ENTREGA:
    print("Falta la entrega en data/fieldshift_datos_v1.2/: sin nombres de condados ni trigo y avena (pasos 2 y 18).")

## 2. Cargar la tabla

`tabla_modelado.csv` tiene una fila por condado, año y cultivo (maíz o soja). El suelo se repite en todas las filas de un condado; el clima cambia cada año.

Si una variable pedida no viene en la tabla, se avisa y se sigue sin ella. No se inventa.

De la entrega se cargan también el nombre y el centroide (latitud y longitud del centro) de cada condado, de `referencia/condados.csv`. El front los usa para escribir el nombre del gemelo y dibujar el arco en el mapa.

In [ ]:
if HAY_DATOS:
    tabla = pd.read_csv(RUTA_TABLA, dtype={"fips": str, "estado": str})
    tabla = tabla[~tabla["anio"].isin(ANIOS_PRUEBA)].copy()

    faltan = [c for c in VARIABLES_SUELO + VARIABLES_CLIMA if c not in tabla.columns]
    if faltan:
        print("No vinieron en la tabla, se sigue sin ellas:", faltan)
    VARIABLES_SUELO = [c for c in VARIABLES_SUELO if c in tabla.columns]
    VARIABLES_CLIMA = [c for c in VARIABLES_CLIMA if c in tabla.columns]

    print(f"{tabla['fips'].nunique()} condados | años {tabla['anio'].min()}–{tabla['anio'].max()}")
    print("Condados por estado:", tabla.groupby("estado")["fips"].nunique().rename(NOMBRES_ESTADO).to_dict())
else:
    print("En espera de la tabla del punto 2.")

CONDADOS = pd.DataFrame(columns=["condado", "lat_centroide", "lon_centroide"])
if HAY_ENTREGA:
    CONDADOS = pd.read_csv(RUTA_ENTREGA / "referencia" / "condados.csv", dtype={"fips": str}).set_index("fips")
    print(f"Nombres y centroides de {len(CONDADOS)} condados")


def nombrar(fips):
    """'Story (Iowa)' si está la entrega; si no, el fips."""
    nombre = CONDADOS["condado"].get(fips)
    return f"{nombre} ({NOMBRES_ESTADO.get(fips[:2], fips[:2])})" if isinstance(nombre, str) else fips

## 3. El perfil de cada condado

Cada condado queda resumido en **una sola fila** con 9 números:

- **Suelo:** el valor del condado, que es el mismo todos los años. El drenaje se pasa de texto a número con `ORDEN_DRENAJE`.
- **Clima típico:** la **mediana** de 2009–2023. La mediana es el valor del medio: la mitad de los años estuvo arriba y la mitad abajo. Se usa en vez del promedio para que un año extremo, como la sequía de 2012, no cambie lo que se considera "el clima de ese condado".

Antes de seguir se chequea que el suelo realmente no cambie entre años: si cambiara, algo raro hay en la tabla.

In [ ]:
if HAY_DATOS:
    tabla["clase_drenaje"] = tabla["clase_drenaje"].map(ORDEN_DRENAJE)

    cambia = tabla.groupby("fips")[VARIABLES_SUELO].nunique().gt(1).any()
    if cambia.any():
        print("Ojo: estas variables de suelo cambian entre años:", cambia[cambia].index.tolist())

    suelo = tabla.groupby("fips")[VARIABLES_SUELO].first()
    # Una fila por condado y año (la tabla repite el clima en maíz y soja), y después la mediana de los años
    clima = tabla.groupby(["fips", "anio"])[VARIABLES_CLIMA].first().groupby("fips").median()
    perfil = suelo.join(clima)
    perfil.insert(0, "estado", perfil.index.str[:2])

    print("Datos faltantes por variable:", perfil.isna().sum()[lambda s: s > 0].to_dict() or "ninguno")
    display(perfil.loc[[f for f in CONDADOS_EJEMPLO if f in perfil.index]])
    display(perfil.describe().T[["mean", "min", "50%", "max"]].round(1))
else:
    print("En espera de la tabla del punto 2.")

## 4. Poner todo en la misma escala

Las variables están en unidades muy distintas: el carbono del suelo anda por los 6.000 g/m² y la productividad del suelo entre 0 y 1. Si las comparamos así, el carbono taparía todo lo demás.

Por eso cada variable se pasa a **cuántos desvíos se aparta del promedio de la región** (*z-score*): 0 es un condado típico, +1 es "bastante más que lo típico" y −1, "bastante menos". Así, una diferencia de 1 significa lo mismo en cualquier variable.

Después se aplica el peso: el suelo tiene 5 variables y el clima 4. Para que pesen lo mismo, cada grupo se divide por la raíz de su cantidad de variables y se multiplica por su peso (`PESO_SUELO`).

In [ ]:
if HAY_DATOS:
    COLUMNAS = VARIABLES_SUELO + VARIABLES_CLIMA
    z = (perfil[COLUMNAS] - perfil[COLUMNAS].mean()) / perfil[COLUMNAS].std()

    pesos = pd.Series(
        {c: np.sqrt(PESO_SUELO / len(VARIABLES_SUELO)) for c in VARIABLES_SUELO}
        | {c: np.sqrt((1 - PESO_SUELO) / len(VARIABLES_CLIMA)) for c in VARIABLES_CLIMA}
    )
    x = z * pesos
    display(z.loc[[f for f in CONDADOS_EJEMPLO if f in z.index]].round(2))
else:
    print("En espera de la tabla del punto 2.")

## 5. La distancia entre cada par de condados

Imaginá un mapa, pero con 9 ejes en vez de 2: uno por variable. Cada condado es un punto en ese mapa. La **distancia** entre dos condados es la línea recta entre sus puntos: cuanto más chica, más parecidos son en suelo y clima. Con 0, serían idénticos.

`nan_euclidean_distances` calcula esa distancia para todos los pares. Si a un condado le falta un dato, compara solo con las variables que tienen los dos y ajusta la escala, en vez de inventar el valor que falta.

Al final se calcula el **umbral de parecido**: la distancia por debajo de la cual está el `PERCENTIL_MAXIMO` % de todos los pares de la región.

In [ ]:
if HAY_DATOS:
    distancias = pd.DataFrame(nan_euclidean_distances(x.to_numpy()), index=x.index, columns=x.index)

    pares = distancias.to_numpy()[np.triu_indices(len(distancias), k=1)]
    UMBRAL = np.percentile(pares, PERCENTIL_MAXIMO)
    print(f"Distancia entre condados: mediana {np.median(pares):.2f}, mínimo {pares.min():.2f}, máximo {pares.max():.2f}")
    print(f"Umbral de parecido (percentil {PERCENTIL_MAXIMO}): {UMBRAL:.2f}")
else:
    print("En espera de la tabla del punto 2.")

## 6. Los más parecidos de cada condado (kNN)

Para cada condado se ordenan todos los demás de menor a mayor distancia y se guardan los primeros `N_VECINOS`, **siempre que estén por debajo del umbral**. Eso es todo el kNN: no aprende nada, solo ordena por distancia.

Un condado no puede ser gemelo de sí mismo, así que se saca de su propia lista.

In [ ]:
if HAY_DATOS:
    filas = []
    for fips in distancias.index:
        cercanos = distancias.loc[fips].drop(fips).sort_values()
        cercanos = cercanos[cercanos <= UMBRAL].head(N_VECINOS)
        for orden, (vecino, dist) in enumerate(cercanos.items(), start=1):
            filas.append({"fips": fips, "orden": orden, "fips_vecino": vecino, "distancia": round(dist, 3)})
    vecinos = pd.DataFrame(filas)
    vecinos["mismo_estado"] = vecinos["fips"].str[:2] == vecinos["fips_vecino"].str[:2]

    cuantos = vecinos.groupby("fips").size().reindex(distancias.index, fill_value=0)
    print(f"Parecidos por condado: mediana {cuantos.median():.0f}, mínimo {cuantos.min()}, máximo {cuantos.max()}")
    print(f"Condados con menos de 3 parecidos (el front mostraría confianza baja): {(cuantos < 3).sum()}")
    print(f"De los parecidos, cuántos son del mismo estado: {vecinos['mismo_estado'].mean():.0%}")
else:
    print("En espera de la tabla del punto 2.")

## 7. En qué se parecen

El front muestra "Similar in: soil water, July heat". Para cada par se toman las `N_SIMILAR_ON` variables en las que la diferencia (en la escala del paso 4) es más chica.

Se guarda en inglés para el front (`similar_on`) y en español para que el equipo revise (`similar_on_es`).

In [ ]:
if HAY_DATOS:
    def calcular_parecido(fips, vecino):
        diferencia = (z.loc[fips] - z.loc[vecino]).abs().dropna().sort_values()
        elegidas = diferencia.index[:N_SIMILAR_ON]
        return [ETIQUETAS[c][0] for c in elegidas], [ETIQUETAS[c][1] for c in elegidas]

    textos = [calcular_parecido(f, v) for f, v in zip(vecinos["fips"], vecinos["fips_vecino"])]
    vecinos["similar_on"] = [", ".join(en) for en, _ in textos]
    vecinos["similar_on_es"] = [", ".join(es) for _, es in textos]
    display(vecinos.head(8))
else:
    print("En espera de la tabla del punto 2.")

## 8. Mirar ejemplos

Para los condados de `CONDADOS_EJEMPLO`: sus 5 parecidos y, al lado, sus valores de suelo y clima. Si el método anda, los números de los parecidos tienen que estar cerca de los del condado.

In [ ]:
if HAY_DATOS:
    for fips in [f for f in CONDADOS_EJEMPLO if f in perfil.index]:
        suyos = vecinos[vecinos["fips"] == fips].head(5)
        comparar = perfil.loc[[fips] + suyos["fips_vecino"].tolist(), COLUMNAS].copy()
        comparar.insert(0, "condado", [nombrar(f) for f in comparar.index])
        comparar.insert(1, "distancia", [0.0] + suyos["distancia"].tolist())
        comparar.insert(2, "similar_on_es", [""] + suyos["similar_on_es"].tolist())
        print(f"\n{nombrar(fips)} y sus 5 parecidos:")
        display(comparar.round(2))
else:
    print("En espera de la tabla del punto 2.")

## 9. ¿El parecido sirve? Chequeo con el rinde

El rinde **no se usó** para buscar los parecidos. Por eso sirve como control: si suelo y clima parecidos implican condiciones parecidas, el rinde típico de maíz de los parecidos tiene que estar más cerca del rinde del condado que el de condados elegidos al azar.

Se comparan tres maneras de elegir 5 condados para cada uno:

- **Parecidos:** los 5 primeros de este punto.
- **Al azar, mismo estado:** 5 condados al azar de su estado. Es la regla simple a superar: "compará con tu estado".
- **Al azar, toda la región.**

Para cada condado se mide la diferencia entre su rinde típico (mediana 2009–2023) y el promedio del de los 5 elegidos. Más chica, mejor. El azar se repite 200 veces con semilla fija para que el resultado no dependa de la suerte.

In [ ]:
if HAY_DATOS:
    rinde = tabla[tabla["cultivo"] == "maiz"].groupby("fips")["rend_bu_ac"].median()
    azar = np.random.default_rng(0)

    def error_contra(grupos):
        """Diferencia media (bu/ac) entre el rinde del condado y el promedio del de su grupo."""
        return np.nanmean([abs(rinde.get(f, np.nan) - rinde.reindex(g).mean()) for f, g in grupos.items()])

    top5 = vecinos[vecinos["orden"] <= 5].groupby("fips")["fips_vecino"].apply(list)
    top5 = top5[top5.str.len() >= 3]
    por_estado = perfil.groupby("estado").groups

    def al_azar(mismo_estado):
        grupos = {}
        for f in top5.index:
            candidatos = [c for c in (por_estado[f[:2]] if mismo_estado else perfil.index) if c != f]
            grupos[f] = list(azar.choice(candidatos, size=5, replace=False))
        return error_contra(grupos)

    resultado = pd.Series({
        "Parecidos (este punto)": error_contra(top5),
        "Al azar, mismo estado": np.mean([al_azar(True) for _ in range(200)]),
        "Al azar, toda la región": np.mean([al_azar(False) for _ in range(200)]),
    }, name="diferencia media de rinde de maíz (bu/ac)")
    print(f"Condados evaluados: {len(top5)} (los que tienen al menos 3 parecidos)")
    display(resultado.round(1).to_frame())
else:
    print("En espera de la tabla del punto 2.")

## 10. Guardar

`data/ml/gemelos_vecinos.csv`: una fila por condado y parecido. Es la entrada de la parte 2.

| Columna | Qué es |
|---|---|
| `fips` | El condado |
| `orden` | 1 = el más parecido |
| `fips_vecino` | El condado parecido |
| `distancia` | Qué tan distintos son en suelo y clima (0 = idénticos) |
| `mismo_estado` | Si están en el mismo estado |
| `similar_on`, `similar_on_es` | Las variables en las que más se parecen |

In [ ]:
if HAY_DATOS:
    ruta = SALIDA / "gemelos_vecinos.csv"
    vecinos.to_csv(ruta, index=False)
    print(f"Guardado {ruta.relative_to(RAIZ)}: {len(vecinos)} filas, {vecinos['fips'].nunique()} condados")
else:
    print("En espera de la tabla del punto 2.")

# Parte 2 · El rango de los gemelos

La pregunta del productor: *"si hago algo que en mi condado casi no se hace, ¿cómo les fue a los condados como el mío que sí lo hacen?"*

Hay dos casos:

1. **Un cultivo que el condado no siembra** (trigo de invierno, avena): paso 18, con la entrega v1.2.
2. **Una rotación que el condado casi no hace** (pasos 11 a 17): todos los condados siembran maíz y soja, pero en algunos casi nadie repite maíz sobre maíz, y en otros casi nadie lo rota. Las opciones son las mismas del punto 2:

| Opción | Qué es | Qué gemelo cuenta como "lo hizo" |
|---|---|---|
| `actual` | Seguir como el condado | Cualquier parecido |
| `rotado` | Casi nada del cultivo sobre sí mismo | Está en el 25 % de condados de la región que **menos** siembra el cultivo sobre sí mismo |
| `continuo` | Mucho del cultivo sobre sí mismo | Está en el 25 % que **más** lo hace |

**Ojo, lo más importante de esta parte:** el rinde de NASS es **del condado entero**, no del lote. Si un gemelo hace mucho maíz sobre maíz, su rinde mezcla esos lotes con los rotados. El rango de los gemelos dice "a los condados como el tuyo que hacen esto les fue así", **no** "si hacés esto vas a rendir así". El paso 16 chequea si las dos cosas coinciden.

## 11. Parámetros de la parte 2

- `ANIO_TENDENCIA`: el rinde sube año a año por la tecnología. Para comparar 2009 con 2023, cada rinde se lleva al nivel de tecnología de 2025, el mismo año que usa el punto 2. Así el rango de los gemelos se puede poner al lado del rango del modelo.
- `CUANTIL_OPCION`: cuánto tiene que hacer un gemelo la opción para contar. 0,25 = estar en el 25 % de la región que más (o menos) lo hace.
- `N_MOSTRAR` y `MIN_GEMELOS`: el front muestra hasta 5 gemelos; con menos de 3, la confianza es baja (regla de `docs/front-end-definido.md`).
- `MIN_ANIOS_ALTA`: para confianza alta, además de 5 gemelos, hace falta un mínimo de años con rinde entre todos.

In [ ]:
ANIO_TENDENCIA = 2025
CUANTIL_OPCION = 0.25
N_MOSTRAR = 5
MIN_GEMELOS = 3
MIN_ANIOS_ALTA = 40

COLUMNA_MISMO = {"maiz": "pct_maiz_tras_maiz", "soja": "pct_soja_tras_soja"}
OPCIONES = ["actual", "rotado", "continuo"]
CULTIVO_EN = {"maiz": "corn", "soja": "soybean"}
# Cómo se nombra, en la frase del front, a los gemelos que hicieron cada opción
OPCION_EN = {
    ("maiz", "actual"): "Counties like yours",
    ("maiz", "rotado"): "Counties like yours that rarely grow corn after corn",
    ("maiz", "continuo"): "Counties like yours that grow a lot of corn after corn",
    ("soja", "actual"): "Counties like yours",
    ("soja", "rotado"): "Counties like yours that rarely grow soybean after soybean",
    ("soja", "continuo"): "Counties like yours that grow a lot of soybean after soybean",
}
RUTA_MODELO_P2 = SALIDA / "escenarios_rotacion_resumen.csv"

## 12. Llevar cada rinde a la tecnología de hoy

Para cada estado y cultivo se ajusta una recta rinde ~ año con 2009–2023 (la **tendencia**). La pendiente dice cuánto sube el rinde por año por la tecnología. Después, a cada rinde se le suma lo que le faltaba para llegar a 2025:

> rinde ajustado = rinde del año + pendiente × (2025 − año)

Ejemplo: si en Iowa el maíz sube 2 bu/ac por año, un rinde de 170 en 2012 pasa a 170 + 2 × 13 = 196. La sequía de 2012 se sigue viendo (queda bajo), pero ya no se confunde con "eran otros tiempos".

No se usa nada de 2024–2025: son la prueba del punto 2.

In [ ]:
if HAY_DATOS:
    rindes = tabla[tabla["rend_bu_ac"].notna()][["fips", "anio", "estado", "cultivo", "rend_bu_ac"]].copy()
    pendientes = {}
    for (cultivo, estado), grupo in rindes.groupby(["cultivo", "estado"]):
        pendiente, _ = np.polyfit(grupo["anio"], grupo["rend_bu_ac"], 1)
        pendientes[(cultivo, estado)] = pendiente
        rindes.loc[grupo.index, "rend_hoy_bu_ac"] = grupo["rend_bu_ac"] + pendiente * (ANIO_TENDENCIA - grupo["anio"])

    display(pd.Series(pendientes, name="bu/ac por año").rename_axis(["cultivo", "estado"]).round(2).to_frame())
else:
    print("En espera de la tabla del punto 2.")

## 13. Qué hace cada condado

Para cada condado y cultivo, el % típico del cultivo sembrado sobre sí mismo (la mediana de 2009–2023). Con eso se calculan los dos cortes:

- **rotado:** el condado está por debajo del cuantil 25 % de la región;
- **continuo:** está por encima del cuantil 75 %.

También se marca si **el propio condado** ya está en esa opción: en ese caso la opción no es nueva para él.

In [ ]:
if HAY_DATOS:
    practica = {}
    cortes = {}
    for cultivo, columna in COLUMNA_MISMO.items():
        pct = tabla[tabla["cultivo"] == cultivo].groupby("fips")[columna].median()
        practica[cultivo] = pct
        cortes[cultivo] = {"rotado": pct.quantile(CUANTIL_OPCION), "continuo": pct.quantile(1 - CUANTIL_OPCION)}
        print(f"{cultivo}: % sobre sí mismo, mediana {pct.median():.1f} %. "
              f"Rotado ≤ {cortes[cultivo]['rotado']:.1f} % | continuo ≥ {cortes[cultivo]['continuo']:.1f} %")

    def hace_la_opcion(cultivo, opcion, fips):
        pct = practica[cultivo].get(fips, np.nan)
        if opcion == "actual":
            return True
        if np.isnan(pct):
            return False
        return pct <= cortes[cultivo]["rotado"] if opcion == "rotado" else pct >= cortes[cultivo]["continuo"]
else:
    print("En espera de la tabla del punto 2.")

## 14. Los gemelos de cada opción

Para cada condado, cultivo y opción se recorren **todos** los condados por debajo del umbral de parecido (paso 5), del más parecido al menos, y se quedan los primeros `N_MOSTRAR` que hicieron la opción.

Se usan todos los que pasan el umbral y no solo los 15 de la parte 1: los condados que hacen mucho maíz sobre maíz suelen estar juntos en una zona, y con solo 15 vecinos muchos condados se quedaban sin ningún gemelo.

De cada gemelo se guarda lo que pide el front: en qué se parece (`similar_on`), cuánto hace la opción (`grew_crop_pct`) y su rinde típico (mediana del rinde llevado a hoy).

In [ ]:
if HAY_DATOS:
    rinde_tipico = rindes.groupby(["cultivo", "fips"])["rend_hoy_bu_ac"].median()

    filas = []
    for fips in distancias.index:
        cercanos = distancias.loc[fips].drop(fips).sort_values()
        cercanos = cercanos[cercanos <= UMBRAL]
        for cultivo in COLUMNA_MISMO:
            for opcion in OPCIONES:
                elegidos = [v for v in cercanos.index if hace_la_opcion(cultivo, opcion, v)][:N_MOSTRAR]
                for orden, vecino in enumerate(elegidos, start=1):
                    en, es = calcular_parecido(fips, vecino)
                    filas.append({
                        "fips": fips, "cultivo": cultivo, "rotacion": opcion, "orden": orden,
                        "fips_gemelo": vecino, "distancia": round(cercanos[vecino], 3),
                        "similar_on": ", ".join(en), "similar_on_es": ", ".join(es),
                        "grew_crop_pct": round(practica[cultivo].get(vecino, np.nan), 1),
                        "rend_tipico_bu_ac": round(rinde_tipico.get((cultivo, vecino), np.nan), 1),
                    })
    gemelos = pd.DataFrame(filas)

    cuantos = (gemelos.groupby(["fips", "cultivo", "rotacion"]).size()
               .reindex(pd.MultiIndex.from_product([distancias.index, list(COLUMNA_MISMO), OPCIONES],
                                                   names=["fips", "cultivo", "rotacion"]), fill_value=0))
    cobertura = cuantos.groupby(["cultivo", "rotacion"]).agg(
        sin_gemelos=lambda s: (s == 0).mean(), menos_de_3=lambda s: (s < MIN_GEMELOS).mean(),
        con_5=lambda s: (s >= N_MOSTRAR).mean())
    print("Proporción de condados según cuántos gemelos tienen:")
    display((cobertura * 100).round(0).astype(int).astype(str) + " %")
else:
    print("En espera de la tabla del punto 2.")

## 15. El rango: p10, mediana y p90

Para cada condado, cultivo y opción se juntan **todos los años de todos sus gemelos** (rinde llevado a hoy). Con 5 gemelos y 15 años son unos 75 valores. De ahí salen:

- **p10, año malo:** solo 1 de cada 10 valores quedó por debajo.
- **mediana, año típico:** la mitad quedó arriba y la mitad abajo.
- **p90, año bueno:** solo 1 de cada 10 quedó por encima.

Es la misma lectura que el rango del punto 2, pero en vez de salir de un modelo sale de lo que **pasó de verdad** en condados parecidos.

**Confianza**, con su motivo en inglés para el front:

- **alta:** 5 gemelos y al menos `MIN_ANIOS_ALTA` años con rinde;
- **media:** 3 o 4 gemelos;
- **baja:** 1 o 2 gemelos;
- **sin dato:** ningún gemelo hizo esa opción. El rango queda vacío y el front muestra "Missing data".

In [ ]:
if HAY_DATOS:
    rinde_por_condado = {k: g["rend_hoy_bu_ac"].to_numpy() for k, g in rindes.groupby(["cultivo", "fips"])}

    def medir_confianza(n_gemelos, n_anios, cultivo, opcion):
        quienes = OPCION_EN[(cultivo, opcion)].replace("Counties like yours", "counties like yours")
        if n_gemelos == 0:
            return "none", "Missing data: no county with similar soil and weather does this."
        if n_gemelos < MIN_GEMELOS:
            if n_gemelos == 1:
                quienes = quienes.replace("counties", "county").replace(" grow ", " grows ")
            return "low", f"Low confidence: only {n_gemelos} {quienes}."
        if n_gemelos >= N_MOSTRAR and n_anios >= MIN_ANIOS_ALTA:
            return "high", f"{n_gemelos} {quienes}, {n_anios} years of yield data."
        return "medium", f"{n_gemelos} {quienes}, {n_anios} years of yield data."

    filas = []
    for (fips, cultivo, opcion), n in cuantos.items():
        suyos = gemelos[(gemelos["fips"] == fips) & (gemelos["cultivo"] == cultivo) & (gemelos["rotacion"] == opcion)]
        valores = np.concatenate([rinde_por_condado.get((cultivo, g), []) for g in suyos["fips_gemelo"]] or [[]])
        nivel, motivo = medir_confianza(n, len(valores), cultivo, opcion)
        filas.append({
            "fips": fips, "cultivo": cultivo, "rotacion": opcion, "n_gemelos": n, "n_anios": len(valores),
            "p10": np.percentile(valores, 10) if len(valores) else np.nan,
            "mediana": np.median(valores) if len(valores) else np.nan,
            "p90": np.percentile(valores, 90) if len(valores) else np.nan,
            "confianza": nivel, "motivo": motivo,
            "condado_ya_lo_hace": hace_la_opcion(cultivo, opcion, fips) if opcion != "actual" else True,
        })
    rango = pd.DataFrame(filas).round({"p10": 1, "mediana": 1, "p90": 1})
    display(rango.groupby(["cultivo", "rotacion"])["confianza"].value_counts(normalize=True)
            .mul(100).round(0).unstack(fill_value=0).astype(int).astype(str) + " %")
else:
    print("En espera de la tabla del punto 2.")

## 16. Chequeo: ¿los gemelos cuentan lo mismo que el modelo del punto 2?

El punto 2 midió el efecto de rotar comparando a cada condado **consigo mismo** (efectos fijos), y da que el cultivo continuo rinde menos, como dicen los experimentos de campo. Los gemelos comparan **condados distintos**. Si las dos cosas apuntan para el mismo lado, el rango de los gemelos se puede mostrar como evidencia de la opción.

Se compara la mediana de `continuo` − la de `rotado`, con los gemelos y con el modelo, en los condados que tienen las dos opciones con al menos 3 gemelos.

Si con los gemelos `continuo` rinde **más** que `rotado`, al revés que el modelo y los experimentos, no se puede mostrar como "lo que te pasaría si hacés esto". Lo más probable es que los condados que hacen mucho maíz sobre maíz sean los más productivos por otras razones (el punto 2 encontró lo mismo antes de usar efectos fijos). En ese caso el cultivo queda marcado con `usar_como_rango = False`: los gemelos se muestran como "quiénes lo hacen", pero el rango de la opción sale del punto 2.

In [ ]:
if HAY_DATOS:
    rango["usar_como_rango"] = rango["confianza"] != "none"
    if RUTA_MODELO_P2.exists():
        modelo_p2 = pd.read_csv(RUTA_MODELO_P2, dtype={"fips": str})
        juntos = rango.merge(modelo_p2, on=["fips", "cultivo", "rotacion"], how="left")
        con_dato = juntos[juntos["n_gemelos"] >= MIN_GEMELOS]

        ancho = con_dato.pivot_table(index=["cultivo", "fips"], columns="rotacion", values=["mediana", "mediana_clima"])
        ancho = ancho.dropna(subset=[("mediana", "continuo"), ("mediana", "rotado")])
        diferencia = pd.DataFrame({
            "gemelos: continuo − rotado": ancho[("mediana", "continuo")] - ancho[("mediana", "rotado")],
            "modelo p2: continuo − rotado": ancho[("mediana_clima", "continuo")] - ancho[("mediana_clima", "rotado")],
        })
        resumen = diferencia.groupby("cultivo").median().round(1)
        resumen["condados"] = diferencia.groupby("cultivo").size()
        display(resumen)

        for cultivo, fila in resumen.iterrows():
            mismo_lado = np.sign(fila["gemelos: continuo − rotado"]) == np.sign(fila["modelo p2: continuo − rotado"])
            print(f"{cultivo}: {'OK, apuntan para el mismo lado' if mismo_lado else 'ATENCIÓN: los gemelos apuntan al revés que el modelo'}")
            if not mismo_lado:
                rango.loc[(rango["cultivo"] == cultivo) & (rango["rotacion"] != "actual"), "usar_como_rango"] = False

        print("\nMediana del rango por opción, gemelos contra modelo del punto 2:")
        display(juntos.groupby(["cultivo", "rotacion"])[["p10", "mediana", "p90", "p10_clima", "mediana_clima", "p90_clima"]]
                .median().round(1))
    else:
        print("No está escenarios_rotacion_resumen.csv del punto 2: el chequeo queda en espera.")
else:
    print("En espera de la tabla del punto 2.")

## 17. Ejemplo

Lo que vería un productor de `CONDADOS_EJEMPLO[0]` (Story, Iowa) en maíz: el rango de cada opción según los gemelos, al lado del rango del modelo (`*_clima`), y los gemelos de la opción que **su condado no hace**.

In [ ]:
if HAY_DATOS:
    fips = CONDADOS_EJEMPLO[0]
    ver = rango[(rango["fips"] == fips) & (rango["cultivo"] == "maiz")]
    if RUTA_MODELO_P2.exists():
        ver = ver.merge(modelo_p2, on=["fips", "cultivo", "rotacion"], how="left")
    display(ver.drop(columns=["fips", "cultivo"]))

    nuevas = ver.loc[~ver["condado_ya_lo_hace"], "rotacion"].tolist()
    if nuevas:
        print(f"Gemelos de la opción '{nuevas[0]}', que {nombrar(fips)} no hace:")
        ejemplo = gemelos[(gemelos["fips"] == fips) & (gemelos["cultivo"] == "maiz") & (gemelos["rotacion"] == nuevas[0])]
        display(ejemplo.assign(gemelo=ejemplo["fips_gemelo"].map(nombrar))
                .drop(columns=["fips", "cultivo", "rotacion", "similar_on"]))
else:
    print("En espera de la tabla del punto 2.")

## 18. Cultivos que el condado no siembra: trigo de invierno y avena

Es el caso más claro de gemelo: *"en mi condado nadie siembra trigo; ¿cómo les fue a los condados como el mío que sí lo siembran?"*. Los rindes vienen de `tabla_anual.csv` de la entrega (NASS, por condado).

- **Quién "siembra" el cultivo:** NASS publica su rinde en al menos `MIN_ANIOS_CULTIVO` de los años 2009–2023. NASS solo publica un condado cuando el cultivo tiene superficie suficiente, así que es una buena señal de que se siembra de verdad. El propio condado se marca con `condado_ya_lo_hace` si lo cumple.
- **`grew_crop_pct`:** qué parte del área sembrada con los cultivos principales (maíz, soja, trigo y avena, superficie sembrada de NASS) fue de ese cultivo, mediana de los años. El CDL trae `pct_trigo`, pero no tiene columna de avena: con NASS los dos se miden igual.
- **Tendencia:** la misma cuenta del paso 12, por estado. Si un estado tiene menos de `MIN_FILAS_TENDENCIA` filas de ese cultivo, se usa la recta de toda la región.
- La opción se llama `nuevo` (id provisorio, falta acordarlo con el front).
- **No hay modelo del punto 2 para estos cultivos**, así que no hay chequeo como el del paso 16. El rango sale solo de los gemelos y vale la misma advertencia: es el rinde de condados que lo siembran, no de tu lote.

In [ ]:
CULTIVOS_NUEVOS = {"trigo_invierno": "winter wheat", "avena": "oats"}
CULTIVOS_AREA = ["maiz", "soja", "trigo_invierno", "avena"]
MIN_ANIOS_CULTIVO = 5
MIN_FILAS_TENDENCIA = 30

if HAY_DATOS and HAY_ENTREGA:
    anual = pd.read_csv(RUTA_ENTREGA / "tabla_anual.csv", dtype={"fips": str},
                        usecols=["fips", "anio", "cultivo", "rend_bu_ac", "sup_sembrada_ac"])
    anual = anual[anual["anio"].between(tabla["anio"].min(), tabla["anio"].max())]
    anual["estado"] = anual["fips"].str[:2]

    area = anual[anual["cultivo"].isin(CULTIVOS_AREA)].pivot_table(
        index=["fips", "anio"], columns="cultivo", values="sup_sembrada_ac")
    total = area.sum(axis=1, min_count=1)

    filas_g, filas_r = [], []
    for cultivo, nombre_en in CULTIVOS_NUEVOS.items():
        CULTIVO_EN[cultivo] = nombre_en
        OPCION_EN[(cultivo, "nuevo")] = f"Counties like yours that grow {nombre_en}"

        d = anual[(anual["cultivo"] == cultivo) & anual["rend_bu_ac"].notna()].copy()
        recta_region = np.polyfit(d["anio"], d["rend_bu_ac"], 1)[0]
        for estado, grupo in d.groupby("estado"):
            pendiente = np.polyfit(grupo["anio"], grupo["rend_bu_ac"], 1)[0] if len(grupo) >= MIN_FILAS_TENDENCIA else recta_region
            d.loc[grupo.index, "rend_hoy_bu_ac"] = grupo["rend_bu_ac"] + pendiente * (ANIO_TENDENCIA - grupo["anio"])

        anios = d.groupby("fips").size()
        siembran = set(anios[anios >= MIN_ANIOS_CULTIVO].index)
        parte = (area[cultivo] / total * 100).groupby("fips").median() if cultivo in area else pd.Series(dtype=float)
        rinde_por_condado.update({(cultivo, f): g["rend_hoy_bu_ac"].to_numpy() for f, g in d.groupby("fips")})
        tipico = d.groupby("fips")["rend_hoy_bu_ac"].median()
        print(f"{nombre_en}: lo siembran {len(siembran)} condados "
              f"({pd.Series([f[:2] for f in siembran]).map(NOMBRES_ESTADO).value_counts().to_dict()})")

        for fips in distancias.index:
            cercanos = distancias.loc[fips].drop(fips).sort_values()
            cercanos = cercanos[cercanos <= UMBRAL]
            elegidos = [v for v in cercanos.index if v in siembran][:N_MOSTRAR]
            for orden, vecino in enumerate(elegidos, start=1):
                en, es = calcular_parecido(fips, vecino)
                filas_g.append({
                    "fips": fips, "cultivo": cultivo, "rotacion": "nuevo", "orden": orden, "fips_gemelo": vecino,
                    "distancia": round(cercanos[vecino], 3), "similar_on": ", ".join(en), "similar_on_es": ", ".join(es),
                    "grew_crop_pct": round(parte.get(vecino, np.nan), 1), "rend_tipico_bu_ac": round(tipico.get(vecino, np.nan), 1),
                })
            valores = np.concatenate([rinde_por_condado.get((cultivo, g), []) for g in elegidos] or [[]])
            nivel, motivo = medir_confianza(len(elegidos), len(valores), cultivo, "nuevo")
            filas_r.append({
                "fips": fips, "cultivo": cultivo, "rotacion": "nuevo", "n_gemelos": len(elegidos), "n_anios": len(valores),
                "p10": np.percentile(valores, 10) if len(valores) else np.nan,
                "mediana": np.median(valores) if len(valores) else np.nan,
                "p90": np.percentile(valores, 90) if len(valores) else np.nan,
                "confianza": nivel, "motivo": motivo, "condado_ya_lo_hace": fips in siembran,
                "usar_como_rango": nivel != "none",
            })

    nuevos_g = pd.DataFrame(filas_g)
    nuevos_r = pd.DataFrame(filas_r).round({"p10": 1, "mediana": 1, "p90": 1})
    gemelos = pd.concat([gemelos[gemelos["rotacion"] != "nuevo"], nuevos_g], ignore_index=True)
    rango = pd.concat([rango[rango["rotacion"] != "nuevo"], nuevos_r], ignore_index=True)

    no_lo_siembran = nuevos_r[~nuevos_r["condado_ya_lo_hace"]]
    print("\nCondados que NO siembran el cultivo, según la confianza de sus gemelos:")
    display((no_lo_siembran.groupby("cultivo")["confianza"].value_counts(normalize=True) * 100)
            .round(0).unstack(fill_value=0).astype(int).astype(str) + " %")

    fips = CONDADOS_EJEMPLO[0]
    print(f"\nEjemplo: {nombrar(fips)}")
    display(nuevos_r[nuevos_r["fips"] == fips].drop(columns=["fips", "rotacion"]))
    ejemplo = nuevos_g[(nuevos_g["fips"] == fips) & (nuevos_g["cultivo"] == "trigo_invierno")]
    display(ejemplo.assign(gemelo=ejemplo["fips_gemelo"].map(nombrar))
            .drop(columns=["fips", "cultivo", "rotacion", "similar_on"]))
elif HAY_DATOS:
    print("En espera de la entrega v1.2: sin trigo ni avena.")
else:
    print("En espera de la tabla del punto 2.")

## 19. Guardar

| Archivo | Qué tiene | Para quién |
|---|---|---|
| `data/ml/gemelos.csv` | Una fila por condado × cultivo × opción × gemelo: `fips_gemelo`, `distancia`, `similar_on`, `grew_crop_pct`, `rend_tipico_bu_ac` | Revisión del equipo |
| `data/ml/gemelos_rango.csv` | Una fila por condado × cultivo × opción: `n_gemelos`, `n_anios`, `p10`, `mediana`, `p90`, `confianza`, `motivo`, `usar_como_rango` | Revisión del equipo |
| `data/ml/front/gemelos.json` | `counties[fips][cultivo][rotacion]` → `twins[]`, `twins_range`, `confidence`, `text`, `use_as_range` | El front (`twins[]` de `next/<fips>.json`) |

En el JSON, cada gemelo trae `name` y `centroid` (`{"lat": ..., "lon": ...}`) de `referencia/condados.csv`. Si falta la entrega, quedan en `null` ("Missing data").

In [ ]:
if HAY_DATOS:
    gemelos.to_csv(SALIDA / "gemelos.csv", index=False)
    rango.to_csv(SALIDA / "gemelos_rango.csv", index=False)

    def numero(v):
        return None if pd.isna(v) else float(v)

    condados = {}
    for _, r in rango.iterrows():
        suyos = gemelos[(gemelos["fips"] == r["fips"]) & (gemelos["cultivo"] == r["cultivo"]) & (gemelos["rotacion"] == r["rotacion"])]
        def centro(f):
            if f not in CONDADOS.index:
                return None
            return {"lat": round(float(CONDADOS.at[f, "lat_centroide"]), 4), "lon": round(float(CONDADOS.at[f, "lon_centroide"]), 4)}

        twins = [{
            "fips": g["fips_gemelo"], "name": CONDADOS["condado"].get(g["fips_gemelo"]),
            "state": NOMBRES_ESTADO.get(g["fips_gemelo"][:2]), "centroid": centro(g["fips_gemelo"]),
            "similar_on": g["similar_on"], "grew_crop_pct": numero(g["grew_crop_pct"]),
            "typical_yield_bu_ac": numero(g["rend_tipico_bu_ac"]),
        } for _, g in suyos.iterrows()]
        quienes = OPCION_EN[(r["cultivo"], r["rotacion"])]
        texto = (f"{quienes}: {r['n_gemelos']}. Their {CULTIVO_EN[r['cultivo']]} yield: {r['p10']:.0f}–{r['p90']:.0f} bu/ac "
                 f"(typical {r['mediana']:.0f}). Similar soil and weather doesn't guarantee the same result."
                 if r["n_gemelos"] else None)
        condados.setdefault(r["fips"], {}).setdefault(r["cultivo"], {})[r["rotacion"]] = {
            "twins": twins,
            "twins_range": {"p10": numero(r["p10"]), "median": numero(r["mediana"]), "p90": numero(r["p90"])},
            "use_as_range": bool(r["usar_como_rango"]),
            "confidence": {"level": r["confianza"], "reason": r["motivo"]},
            "text": texto, "basis": "data", "source": "nass", "source_label": "USDA NASS + CDL, NASA POWER, SSURGO",
        }

    documento = {
        "version_contrato": "0.1", "generado": datetime.now().isoformat(timespec="seconds"),
        "es_prueba": False, "prototype": False, "metric": "yield_bu_ac", "tech_year": ANIO_TENDENCIA,
        "note": ("Counties with similar soil and weather (2009–2023). Yield is for the whole county, not a single field. "
                 "When use_as_range is false, show the twins as who does it, and take the option's range from the model."),
        "counties": condados,
    }
    (SALIDA / "front").mkdir(exist_ok=True)
    with open(SALIDA / "front" / "gemelos.json", "w", encoding="utf-8") as f:
        json.dump(documento, f, ensure_ascii=False, indent=1)
    print(f"Guardados gemelos.csv ({len(gemelos)} filas), gemelos_rango.csv ({len(rango)} filas) y front/gemelos.json")
else:
    print("En espera de la tabla del punto 2.")

## Próximos pasos

- [ ] Decidir con el equipo qué hacer con `use_as_range = False` (maíz): mostrar los gemelos solo como "quiénes lo hacen" o buscar otra forma de comparar.
- [ ] Acordar con el front el id `nuevo` para los cultivos que el condado no siembra, y si el centroide va como `{"lat", "lon"}`.
- [ ] Kansas (Sumner) no está en la entrega v1.2: hasta que llegue, sin gemelos para Sumner.
- [ ] Acordar con el equipo las variables y el peso del parecido (decisión abierta en `docs/front-end-definido.md`).
- [ ] Escribir `PUNTO5.md` y la fila del punto 5 en `README.md`.